# SQLite Setup


A SQLite database with tables corresponding to each table in the [MBTA GTFS Documenation](https://github.com/mbta/gtfs-documentation/blob/master/reference/gtfs.md) has been loaded for your convenience. Data exploration will be performed in SQL throughout this guide. 

In [1]:
.open feed.db

You can now execute SQL queries on the database.

In [2]:
-- Select this cell and press Shift + Return

SELECT *
FROM
	feed_info;

feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_id
MBTA,http://www.mbta.com,EN,20240127,20240406,"Winter 2024, 2024-02-03T02:07:36+00:00, version D",developer@mbta.com,mbta-ma-us


# Finding stops visited twice

Finding stops visited twice that are visited twice on a trip is defined by finding trips in which a `stop_id` appears twice in the `stop_times`.
This can be done by finding the groups of `(stop_id, trip_id)` for each `route_patterns.representative_trip_id`, where the size of the group is larger than `1`, meaning that the `stop_id` was visited `COUNT` times.

In [3]:
SELECT DISTINCT
	stop_time.stop_id,

	stop.stop_name,

	COUNT(stop_time.stop_id)
		AS total_stops,

	group_concat(stop_time.stop_sequence)
		AS stop_sequences,

	trip.route_id,
	trip.trip_headsign,
	trip.route_pattern_id,
	trip.shape_id,
	trip.trip_id
		AS representative_trip_id

FROM
	stop_times
		AS stop_time
JOIN
	stops
		AS stop
	ON
		stop_time.stop_id
			= stop.stop_id,

	trips
		AS trip
	ON
		trip.trip_id = stop_time.trip_id


WHERE
	stop_time.trip_id IN (
		SELECT
			pattern.representative_trip_id
		FROM
			route_patterns
				AS pattern
	)

GROUP BY
	stop_time.trip_id,
	stop_time.stop_id

HAVING
	total_stops
		> 1

LIMIT 10

stop_id,stop_name,total_stops,stop_sequences,route_id,trip_headsign,route_pattern_id,shape_id,representative_trip_id
838,Lagrange St @ Vermont St,2,"4,13",37,Avenue Louis Pasteur via Forest Hills,37-3-1,370132-1,60451996_1
5881,Bennington St @ Antrim St,2,"18,35",120,Orient Heights via Wood Island,120-2-0,1200148,60467724
3125,North Quincy,2,"48,54",217,Quincy Center via North Quincy,217-1-0,2170118,60487527
4023,Granite St @ Davis Rd,2,"33,36",238,Holbrook/Randolph,238-9-0,2380227,60488003
4023,Granite St @ Davis Rd,2,"33,36",238,Crawford Square,238-7-0,2380230,60488004
4023,Granite St @ Davis Rd,2,"33,36",238,Avon Square,238-3-0,2380231,60488034
